In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df_news_final_project = pd.read_parquet(
    "/content/drive/MyDrive/NLP_AI_Impact_Project/data/cleaned_data.parquet"
)

print(df_news_final_project.shape)
df_news_final_project.head()

(188910, 8)


,url,date,language,title,text,word_count,clean_text,processed_text
0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod...",483,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",bad idea ai price bad market cap price today c...
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,\n\nThis AI video of gymnastics might be the f...,812,\n\nThis AI video of gymnastics might be the f...,this ai video of gymnastics might be the frea...
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...","\n\nIf using AI feels like a chore, try this -...",884,"\n\nIf using AI feels like a chore, try this -...",if using ai feels like a chore try this boing...
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,en,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation M...,596,The Road Ahead: How China's AI Foundation M...,the road ahead how china s ai foundation mode...
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,en,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers ...,622,Microsoft and Nvidia to Empower Developers ...,microsoft and nvidia to empower developers wi...


In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text

custom_stopwords = [
    "news","said","new","home","share","watch","video","menu",
    "subscribe","search","content","business","market",
    "opens","open","account","login","cookies","local",
    "world","global","india","united"
]

all_stopwords = list(text.ENGLISH_STOP_WORDS.union(custom_stopwords))

vectorizer = CountVectorizer(
    max_features=5000,
    stop_words=all_stopwords,
    min_df=20,
    max_df=0.7
)

X = vectorizer.fit_transform(df_news_final_project["processed_text"])

In [4]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 12

lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch"
)

lda.fit(X)

LatentDirichletAllocation(n_components=12, random_state=42)

In [5]:
import numpy as np

def display_topics(model, feature_names, n_top_words=15):
    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic {topic_idx}:")
        print(" ".join([feature_names[i]
                        for i in topic.argsort()[:-n_top_words - 1:-1]]))

feature_names = vectorizer.get_feature_names_out()

display_topics(lda, feature_names)


Topic 0:
nasdaq stocks stock nvidia markets data company technology earnings solutions companies growth year financial investing

Topic 1:
price rate trading usd crypto rates mexc buy time real trends exchange change value conversion

Topic 2:
services overviewview products technology consumer ment policy media general health entertain public financial computer transportation

Topic 3:
weather sports ago health live skip stories politics hours entertainment school day alert tv national

Topic 4:
data research technology intelligence industry artificial press digital growth company report information energy innovation government

Topic 5:
best crypto bitcoin million buy online billion blockchain token betting mexc review trading investment investors

Topic 6:
best music smart schedule mobile radio gaming tv npr public reviews lg support air games

Topic 7:
newswires south north carolina dakota virginia guinea republic state mexico country georgia industry international distribution

To

In [6]:
topic_distribution = lda.transform(X)
df_news_final_project["topic"] = topic_distribution.argmax(axis=1)

df_news_final_project["topic"].value_counts()

,count
topic,
3,45955
9,33586
10,28341
4,23366
6,11971
0,10615
8,10025
11,9011
5,4785


In [10]:
industry_dict = {
    "Healthcare": [
        "hospital", "medical", "healthcare", "pharma",
        "biotech", "clinical", "patient", "drug",
        "diagnostic", "surgery", "therapy"
    ],
    "Finance": [
        "bank", "financial", "investment", "insurance",
        "trading", "hedge fund", "asset management",
        "crypto", "blockchain", "fintech"
    ],
    "Legal": [
        "law", "legal", "attorney", "court",
        "litigation", "compliance", "regulation"
    ],
    "Manufacturing": [
        "manufacturing", "factory", "industrial",
        "robotics", "supply chain", "assembly"
    ],
    "Education": [
        "education", "school", "university",
        "students", "teacher", "curriculum",
        "learning platform"
    ],
    "Government": [
        "government", "federal", "policy",
        "regulation", "public sector",
        "defense", "military"
    ],
    "Retail": [
        "retail", "ecommerce", "consumer",
        "shopping", "merchandise",
        "marketplace"
    ],
    "Energy": [
        "energy", "oil", "gas",
        "renewable", "utilities",
        "solar", "wind"
    ],
    "Transportation": [
        "transportation", "automotive",
        "logistics", "airline", "shipping",
        "autonomous vehicle", "self-driving"
    ],
    "Tech Platforms": [
        "cloud", "software", "platform",
        "saas", "enterprise", "data center",
        "semiconductor"
    ],
    "Media & Entertainment": [
        "media", "entertainment", "gaming",
        "film", "music", "streaming",
        "broadcast", "advertising"
    ],
    "Telecommunications": [
        "telecom", "telecommunications",
        "5g", "network operator"
    ],
    "Real Estate": [
        "real estate", "property",
        "housing", "mortgage"
    ],
    "Insurance": [
        "insurance", "insurer", "underwriting"
    ]
}

In [11]:
def assign_industry(text):
    text_lower = text.lower()
    scores = {}

    for industry, keywords in industry_dict.items():
        scores[industry] = sum(
            keyword in text_lower for keyword in keywords
        )

    best_industry = max(scores, key=scores.get)

    if scores[best_industry] == 0:
        return "Other"

    return best_industry

df_news_final_project["industry"] = (
    df_news_final_project["clean_text"]
    .astype(str)
    .apply(assign_industry)
)

df_news_final_project["industry"].value_counts()

,count
industry,
Media & Entertainment,32776
Finance,32112
Other,31223
Education,21226
Tech Platforms,19007
Legal,15348
Healthcare,12123
Government,7039
Energy,5468


In [12]:
df_news_final_project.to_parquet(
    "/content/drive/MyDrive/NLP_AI_Impact_Project/data/industry_labeled.parquet"
)